# Notebook 22 — Discretion Intensity

Becker competitive-discipline test.

**Input:** panel files  **Output:** table_22, figure_22  **Runtime:** ~30 min

In [ ]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

PROC = Path("../data/processed")
TABS = Path("../outputs/tables"); TABS.mkdir(exist_ok=True)
FIGS = Path("../outputs/figures"); FIGS.mkdir(exist_ok=True)
YEARS = [2020,2021,2022,2023,2024]; BLACK_CODE = 3
MIN_B = 20; MIN_W = 20

print("="*65)
print("NB22: DISCRETION INTENSITY — Lender Size and Market HHI")
print("="*65)
print("Theory: Becker (1957) — competitive discipline constrains")
print("differential treatment. Gap should be larger where competition")
print("is weaker: small lenders, concentrated markets.")
print()

dfs = []
for yr in YEARS:
    fp = PROC / f"panel_{yr}.csv"
    if not fp.exists():
        print(f"  {yr}: missing")
        continue
    df = pd.read_csv(fp, usecols=["lei","applicant_race_1","approved",
                                   "income","loan_amount","ltv","state_code"])
    df = df[df["applicant_race_1"].isin([BLACK_CODE,5])].copy()
    df["black"]    = (df["applicant_race_1"]==BLACK_CODE).astype(int)
    df["approved"] = pd.to_numeric(df["approved"], errors="coerce")
    df["income"]   = pd.to_numeric(df["income"],   errors="coerce")
    df["ltv"]      = pd.to_numeric(df["ltv"],      errors="coerce")
    df["year"] = yr
    dfs.append(df.dropna(subset=["approved","income","ltv"]))
    print(f"  {yr}: {len(dfs[-1]):,} obs")

df_all = pd.concat(dfs, ignore_index=True)
print(f"Total: {len(df_all):,}")


In [ ]:

# Within-lender gap per lender-year
lender_gaps = []
for (lei, yr), grp in df_all.groupby(["lei","year"]):
    b = grp[grp["black"]==1]; w = grp[grp["black"]==0]
    if len(b)<MIN_B or len(w)<MIN_W:
        continue
    gap = (w["approved"].mean()-b["approved"].mean())*100
    state = grp["state_code"].iloc[0] if "state_code" in grp.columns else None
    lender_gaps.append({"lei":lei,"year":yr,"gap_pp":gap,
                        "n_total":len(grp),"state_code":state})

df_lender = pd.DataFrame(lender_gaps)
sz = df_all.groupby("lei").size().reset_index(name="total_apps")
df_lender = df_lender.merge(sz, on="lei", how="left")
print(f"Lender-years: {len(df_lender):,}  Mean gap: {df_lender['gap_pp'].mean():.2f}pp")

# --- Size quartiles ---
df_lender["size_q"] = pd.qcut(df_lender["total_apps"], q=4,
    labels=["Q1 Small","Q2","Q3","Q4 Large"])
size_res = []
for q in ["Q1 Small","Q2","Q3","Q4 Large"]:
    sub = df_lender[df_lender["size_q"]==q]
    m = sub["gap_pp"].mean()
    se = sub["gap_pp"].std()/np.sqrt(len(sub))
    size_res.append({"Group":q,"Dim":"Size","N":len(sub),
                     "Mean_gap":round(m,3),"SE":round(se,3),
                     "CI_L":round(m-1.96*se,3),"CI_U":round(m+1.96*se,3),
                     "Median_apps":int(sub["total_apps"].median())})
    print(f"  {q}: {m:.2f}pp  SE={se:.3f}  N={len(sub):,}")

# --- HHI ---
state_hhi = []
for state, grp in df_all.groupby("state_code"):
    counts = grp.groupby("lei").size()
    hhi    = ((counts/counts.sum())**2).sum()*10000
    state_hhi.append({"state_code":state,"hhi":hhi})

df_hhi_st = pd.DataFrame(state_hhi)
df_lender  = df_lender.merge(df_hhi_st, on="state_code", how="left")
dl_h = df_lender.dropna(subset=["hhi"]).copy()
dl_h["hhi_q"] = pd.qcut(dl_h["hhi"], q=4,
    labels=["Q1 Low (competitive)","Q2","Q3","Q4 High (concentrated)"])

hhi_res = []
for q in ["Q1 Low (competitive)","Q2","Q3","Q4 High (concentrated)"]:
    sub = dl_h[dl_h["hhi_q"]==q]
    m = sub["gap_pp"].mean()
    se = sub["gap_pp"].std()/np.sqrt(len(sub))
    hhi_res.append({"Group":q,"Dim":"HHI","N":len(sub),
                    "Mean_gap":round(m,3),"SE":round(se,3),
                    "CI_L":round(m-1.96*se,3),"CI_U":round(m+1.96*se,3),
                    "Median_HHI":round(sub["hhi"].median(),0)})
    print(f"  {q}: {m:.2f}pp  SE={se:.3f}  N={len(sub):,}")

df_out = pd.DataFrame(size_res + hhi_res)
df_out.to_csv(TABS/"table_22_discretion_intensity.csv", index=False)
print("Saved table_22_discretion_intensity.csv")


In [ ]:

fig, axes = plt.subplots(1,2,figsize=(14,6))
for ax, res, title in [
    (axes[0], size_res, "Panel A: Gap by Lender Size Quartile"),
    (axes[1], hhi_res,  "Panel B: Gap by Market HHI Quartile")]:
    labels = [r["Group"] for r in res]
    means  = [r["Mean_gap"] for r in res]
    cis    = [1.96*r["SE"] for r in res]
    colors = ["#90CAF9","#42A5F5","#1976D2","#0D47A1"]
    ax.bar(range(len(labels)), means, color=colors, alpha=0.85, edgecolor="white")
    ax.errorbar(range(len(labels)), means, yerr=cis, fmt="none",
                color="black", capsize=6, capthick=2)
    for i,(m,c) in enumerate(zip(means,cis)):
        ax.text(i, m+c+0.15, f"{m:.1f}pp", ha="center", fontsize=9, fontweight="bold")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=8, rotation=10, ha="right")
    ax.set_ylabel("Within-Lender Racial Gap (pp)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.grid(alpha=0.3, axis="y")
fig.suptitle("Figure 22: Discretion Intensity\nWithin-lender racial gap by lender size and market concentration",
    fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS/"figure_22_discretion.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved figure_22_discretion.png")
print("NB22 COMPLETE")
